In [1]:
import sys
from pathlib import Path
import numpy as np
import cv2
import torch
import json

REPO_ROOT = Path("../..").resolve()
PNL_DIR = REPO_ROOT / "external" / "PnLCalib"

# Añadir PnLCalib al path para poder importar sus módulos
sys.path.insert(0, str(PNL_DIR))

# Importar las piezas de PnLCalib que necesitamos
from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l
from utils.utils_calib import FramebyFrameCalib
from utils.utils_heatmap import (get_keypoints_from_heatmap_batch_maxpool,
                                  get_keypoints_from_heatmap_batch_maxpool_l,
                                  complete_keypoints, coords_to_dict)
import torchvision.transforms as T
import torchvision.transforms.functional as f_tv
from PIL import Image
import yaml

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PnLCalib en path: {PNL_DIR.exists()}")

Device: cuda
PnLCalib en path: True


In [2]:
# Cargar los dos modelos (keypoints y líneas) con sus configs
cfg_kp = yaml.safe_load(open(PNL_DIR / "config" / "hrnetv2_w48.yaml"))
cfg_l  = yaml.safe_load(open(PNL_DIR / "config" / "hrnetv2_w48_l.yaml"))

model = get_cls_net(cfg_kp)
model.load_state_dict(torch.load(PNL_DIR / "weights" / "SV_kp", map_location=device))
model.to(device).eval()

model_l = get_cls_net_l(cfg_l)
model_l.load_state_dict(torch.load(PNL_DIR / "weights" / "SV_lines", map_location=device))
model_l.to(device).eval()

transform = T.Resize((540, 960))
print("Modelos de PnLCalib cargados")

Modelos de PnLCalib cargados


In [8]:
def homografia_imagen_a_campo(final_params_dict):
    """Convierte los parámetros de cámara de PnLCalib en una homografía píxel->campo (metros),
    con origen en la ESQUINA del campo (tu sistema), no en el centro (el de PnLCalib).
    Devuelve None si la calibración no es válida."""
    if final_params_dict is None or "cam_params" not in final_params_dict:
        return None
    from inference import projection_from_cam_params
    P = projection_from_cam_params(final_params_dict)   # 3x4, campo(centro,Z=0)->imagen

    # Homografía suelo(Z=0)->imagen = columnas 0,1,3 de P
    H_campo_a_img = P[:, [0, 1, 3]]
    try:
        H_img_a_campo = np.linalg.inv(H_campo_a_img)
    except np.linalg.LinAlgError:
        return None

    # PnLCalib usa origen en el CENTRO (resta 105/2, 68/2); tu pipeline usa la esquina.
    T_centro_a_esquina = np.array([[1, 0, 52.5],
                                   [0, 1, 34.0],
                                   [0, 0, 1.0]])
    H_final = T_centro_a_esquina @ H_img_a_campo
    return H_final / H_final[2, 2]

print("Función homografia_imagen_a_campo definida")

# Probarla ahora mismo con el 'params' que ya tienes del frame 0
H = homografia_imagen_a_campo(params)
print(f"¿Homografía obtenida? {H is not None}")
if H is not None:
    for (px, py, etiqueta) in [(960, 864, "centro-bajo"), (960, 540, "centro"), (300, 800, "izq-bajo")]:
        campo = cv2.perspectiveTransform(np.array([[[px, py]]], dtype=np.float32), H)[0][0]
        print(f"  pixel ({px},{py}) [{etiqueta}] -> campo ({campo[0]:.1f}, {campo[1]:.1f}) m")
    print("  Esperado: valores dentro de [0,105]x[0,68]. El frame 0 mira al área izquierda,")
    print("  así que X bajos (zona izquierda del campo) tienen sentido.")

Función homografia_imagen_a_campo definida
¿Homografía obtenida? True
  pixel (960,864) [centro-bajo] -> campo (33.3, 58.0) m
  pixel (960,540) [centro] -> campo (26.4, 37.0) m
  pixel (300,800) [izq-bajo] -> campo (17.8, 59.3) m
  Esperado: valores dentro de [0,105]x[0,68]. El frame 0 mira al área izquierda,
  así que X bajos (zona izquierda del campo) tienen sentido.


In [9]:
import inference as pnl_inference
import torchvision.transforms as T
import torch

# Inyectar TODAS las variables globales que inference.py define en su main
# y que sus funciones internas usan (transform2, transform, device).
device = "cuda" if torch.cuda.is_available() else "cpu"
pnl_inference.device = device
pnl_inference.transform2 = T.Resize((540, 960))
pnl_inference.transform = T.Resize((540, 960))

from inference import inference

VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"

cap = cv2.VideoCapture(str(VIDEO_IN))
ret, frame0 = cap.read()
cap.release()
assert ret, "No se pudo leer el primer frame"

h_img, w_img = frame0.shape[:2]
print(f"Frame: {w_img}x{h_img}  |  device: {device}")

cam = FramebyFrameCalib(iwidth=w_img, iheight=h_img, denormalize=True)
KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE = 0.1486, 0.3880, True

params = inference(cam, frame0, model, model_l, KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE)

print(f"\n¿Devolvió parámetros? {params is not None}")
if params is not None:
    print(f"Claves: {list(params.keys())}")
    if "cam_params" in params:
        print(f"cam_params: {list(params['cam_params'].keys())}")
    H = homografia_imagen_a_campo(params)
    print(f"\n¿Homografía obtenida? {H is not None}")
    if H is not None:
        centro_img = np.array([[[w_img/2, h_img*0.8]]], dtype=np.float32)
        campo = cv2.perspectiveTransform(centro_img, H)[0][0]
        print(f"Pixel ({w_img/2:.0f},{h_img*0.8:.0f}) -> campo ({campo[0]:.1f}, {campo[1]:.1f}) m")
        print(f"  (debería caer dentro de ~[0,105]x[0,68] si es buena)")

Frame: 1920x1080  |  device: cuda

¿Devolvió parámetros? True
Claves: ['mode', 'use_ransac', 'rep_err', 'cam_params', 'calib_plane']
cam_params: ['pan_degrees', 'tilt_degrees', 'roll_degrees', 'x_focal_length', 'y_focal_length', 'principal_point', 'position_meters', 'rotation_matrix', 'radial_distortion', 'tangential_distortion', 'thin_prism_distortion']

¿Homografía obtenida? True
Pixel (960,864) -> campo (33.3, 58.0) m
  (debería caer dentro de ~[0,105]x[0,68] si es buena)


In [10]:
import inference as pnl_inference
import torchvision.transforms as T
from inference import inference
from tqdm import tqdm

# Asegurar las globales de PnLCalib (por si se reinició el kernel)
pnl_inference.device = device
pnl_inference.transform2 = T.Resize((540, 960))
pnl_inference.transform = T.Resize((540, 960))

VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"
NOMBRE = "A1606b0e6_0_62"
OUT_H = REPO_ROOT / "data" / "tracking" / f"homografias_pnl_{NOMBRE}.npz"
OUT_H.parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_IN))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w_img = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_img = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cam = FramebyFrameCalib(iwidth=w_img, iheight=h_img, denormalize=True)
KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE = 0.1486, 0.3880, True

homografias = {}
n_ok = n_fail = 0

for f_idx in tqdm(range(n_frames)):
    ret, frame = cap.read()
    if not ret:
        break
    try:
        params = inference(cam, frame, model, model_l, KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE)
        H = homografia_imagen_a_campo(params)
    except Exception:
        H = None
    if H is not None:
        homografias[f_idx] = H
        n_ok += 1
    else:
        n_fail += 1
cap.release()

arr = np.full((n_frames, 3, 3), np.nan)
for f_idx, H in homografias.items():
    arr[f_idx] = H
np.savez_compressed(OUT_H, homografias=arr, n_frames=n_frames)

print(f"\nFrames calibrados: {n_ok}/{n_frames} ({100*n_ok/n_frames:.0f}%)")
print(f"Frames sin calibración: {n_fail}")
print(f"Guardado en: {OUT_H}")

100%|██████████| 749/749 [07:36<00:00,  1.64it/s]


Frames calibrados: 749/749 (100%)
Frames sin calibración: 0
Guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\tracking\homografias_pnl_A1606b0e6_0_62.npz
